# CS535 Project 2 — IMDb：從 1D CNN、DistilBERT 到 PLM 思維

這份 notebook 刻意比一般 sentiment tutorial 詳細，因為目標不是只做出 accuracy，而是建立之後做蛋白質/短胜肽預測需要的完整概念：

1. dataset 與科學化 split，
2. TF-IDF baseline，
3. token ID 與 learned embedding，
4. 可以手算的 Conv1D，
5. single-window TextCNN，
6. kernel size / filter number controlled experiments，
7. multi-window CNN，
8. filter activation interpretation，
9. DistilBERT tokenizer 與 hidden states，
10. frozen DistilBERT + MCNN，
11. optional fine-tuning，
12. error analysis，
13. 一對一映射到 Protein Language Model。

**研究規則：**validation 用於 model selection；official IMDb test set 在最後比較前不要使用。

In [ ]:
!pip -q install datasets transformers accelerate scikit-learn pandas matplotlib

In [ ]:
import re, random, time, math, os
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix)

SEED=535
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FAST_MODE=True
print('Device:',DEVICE)

## 1. 載入 IMDb，並保留真正 independent test

Hugging Face IMDb 包含 25,000 labeled training reviews、25,000 labeled test reviews 與 50,000 unlabeled reviews。我們只從 official train 切出 validation，official test 留到最後。

In [ ]:
imdb=load_dataset('imdb')
train_raw=imdb['train'].train_test_split(test_size=0.20, seed=SEED, stratify_by_column='label')
train_ds, val_ds = train_raw['train'], train_raw['test']
test_ds = imdb['test']
if FAST_MODE:
    train_ds=train_ds.shuffle(seed=SEED).select(range(8000))
    val_ds=val_ds.shuffle(seed=SEED).select(range(2000))
    test_ds=test_ds.shuffle(seed=SEED).select(range(3000))
print(len(train_ds),len(val_ds),len(test_ds))
print(pd.Series(train_ds['label']).value_counts())
print(train_ds[0]['text'][:700], '\nlabel=', train_ds[0]['label'])

## 2. 基本 EDA：class balance 與 review length

長度很重要，因為神經網路需要選 max sequence length。Truncation 是模型設計決策，不只是程式細節。

In [ ]:
TOKEN_RE=re.compile(r"[A-Za-z']+")
def simple_tokenize(x): return TOKEN_RE.findall(x.lower())
lengths=np.array([len(simple_tokenize(x)) for x in train_ds['text']])
print(pd.Series(lengths).describe(percentiles=[.5,.75,.9,.95,.99]))
plt.figure(figsize=(8,4)); plt.hist(np.clip(lengths,0,1000),bins=60); plt.xlabel('tokens (clipped at 1000)'); plt.ylabel('reviews'); plt.show()

## 3. 非神經網路 baseline：TF-IDF + Logistic Regression

複雜模型應該先跟合理的簡單 baseline 比。如果連 baseline 都無法穩定超越，就不能只因模型較新而宣稱有價值。

In [ ]:
tfidf=TfidfVectorizer(max_features=30000,ngram_range=(1,2),min_df=2)
Xtr=tfidf.fit_transform(train_ds['text']); Xva=tfidf.transform(val_ds['text'])
logreg=LogisticRegression(max_iter=1000,n_jobs=None)
logreg.fit(Xtr,train_ds['label'])
p=logreg.predict_proba(Xva)[:,1]
print('validation accuracy',accuracy_score(val_ds['label'],p>=.5),'ROC-AUC',roc_auc_score(val_ds['label'],p))

## 4. 建 vocabulary 與 learned embedding baseline

Token ID 只是整數索引；Embedding layer 才把每個 ID 轉成可學習的 dense vector。

In [ ]:
counter=Counter()
for t in train_ds['text']: counter.update(simple_tokenize(t))
MAX_VOCAB=30000
itos=['<pad>','<unk>']+[w for w,_ in counter.most_common(MAX_VOCAB-2)]
stoi={w:i for i,w in enumerate(itos)}
PAD,UNK=0,1
MAX_LEN=300

def encode(text):
    ids=[stoi.get(w,UNK) for w in simple_tokenize(text)[:MAX_LEN]]
    ids += [PAD]*(MAX_LEN-len(ids))
    return torch.tensor(ids,dtype=torch.long)
print('vocab',len(itos),'example',simple_tokenize(train_ds[0]['text'])[:20])

## 5. 手算 Conv1D

在訓練任何模型前，先真的算一次 convolution。這就是前面 2D convolution 手算在 sequence 上的對應版本。

In [ ]:
X=torch.tensor([[1.,0.],[1.,1.],[0.,1.],[0.,2.],[1.,2.]])  # L=5,D=2
W=torch.tensor([[1.,0.],[0.,1.],[-1.,1.]])               # k=3,D=2
scores=[]
for i in range(X.shape[0]-W.shape[0]+1):
    patch=X[i:i+3]
    score=(patch*W).sum()
    scores.append(float(score))
    print(f'position {i}:')
    print('patch=\n',patch.numpy())
    print('elementwise=\n',(patch*W).numpy(),' sum=',float(score))
print('feature map:',scores)

## 6. PyTorch Conv1D 的 tensor shapes

Embedding 通常是 `[B,L,D]`；Conv1D 要的是 `[B,D,L]`。這一步一定要讓學生自己印 shape。

In [ ]:
demo_ids=torch.randint(0,len(itos),(4,20))
emb=nn.Embedding(len(itos),128)(demo_ids)
print('embedding:',emb.shape)
x=emb.transpose(1,2)
print('for Conv1D:',x.shape)
conv=nn.Conv1d(128,64,kernel_size=3)
y=conv(x)
print('Conv output:',y.shape)
pooled=F.adaptive_max_pool1d(y,1).squeeze(-1)
print('pooled:',pooled.shape)

In [ ]:
class WordDS(Dataset):
    def __init__(self,ds): self.ds=ds
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        r=self.ds[i]; return encode(r['text']),torch.tensor(float(r['label']))
def make_loader(ds,bs=64,shuffle=False): return DataLoader(WordDS(ds),batch_size=bs,shuffle=shuffle)
tr_loader=make_loader(train_ds,64,True); va_loader=make_loader(val_ds); te_loader=make_loader(test_ds)

## 7. Single-window TextCNN

先只用一個 kernel size，讓 window length 的角色完全清楚。

In [ ]:
class SingleTextCNN(nn.Module):
    def __init__(self,vocab,emb_dim=128,k=3,filters=128,dropout=.3):
        super().__init__(); self.embedding=nn.Embedding(vocab,emb_dim,padding_idx=PAD)
        self.conv=nn.Conv1d(emb_dim,filters,k); self.drop=nn.Dropout(dropout); self.fc=nn.Linear(filters,1)
    def forward(self,ids,return_map=False):
        x=self.embedding(ids).transpose(1,2); fmap=F.relu(self.conv(x))
        pooled=F.adaptive_max_pool1d(fmap,1).squeeze(-1); out=self.fc(self.drop(pooled)).squeeze(1)
        return (out,fmap) if return_map else out

In [ ]:
def metric_dict(y,p):
    pred=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),specificity=tn/(tn+fp),f1=f1_score(y,pred,zero_division=0),mcc=matthews_corrcoef(y,pred),roc_auc=roc_auc_score(y,p),pr_auc=average_precision_score(y,p))
def eval_word(model,loader):
    model.eval();ys=[];ps=[]
    with torch.no_grad():
        for x,y in loader:
            p=torch.sigmoid(model(x.to(DEVICE))).cpu().numpy(); ys.extend(y.numpy());ps.extend(p)
    return np.array(ys),np.array(ps)
def fit_word(model,epochs=3,lr=1e-3):
    model.to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=lr);lossfn=nn.BCEWithLogitsLoss();best=None;best_auc=-1
    for ep in range(1,epochs+1):
        model.train()
        for x,y in tr_loader:
            x,y=x.to(DEVICE),y.to(DEVICE);opt.zero_grad(set_to_none=True);loss=lossfn(model(x),y);loss.backward();opt.step()
        yy,pp=eval_word(model,va_loader);m=metric_dict(yy,pp);print(ep,m)
        if m['roc_auc']>best_auc: best_auc=m['roc_auc'];best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best); return model
single=fit_word(SingleTextCNN(len(itos),k=3),epochs=2 if FAST_MODE else 6)

## 8. Kernel-size controlled experiment

一次只改 window size；representation、filters、optimizer、split、training protocol 都固定。

In [ ]:
kernel_rows=[]
for k in [2,3,4,5,7]:
    m=fit_word(SingleTextCNN(len(itos),k=k,filters=64),epochs=1 if FAST_MODE else 4)
    y,p=eval_word(m,va_loader);kernel_rows.append({'window':k,**metric_dict(y,p)})
display(pd.DataFrame(kernel_rows))

## 9. Filter-count controlled experiment

In [ ]:
filter_rows=[]
for f in [32,64,128,256]:
    m=fit_word(SingleTextCNN(len(itos),k=3,filters=f),epochs=1 if FAST_MODE else 4)
    y,p=eval_word(m,va_loader);filter_rows.append({'filters':f,**metric_dict(y,p)})
display(pd.DataFrame(filter_rows))

## 10. Multi-window CNN

平行 windows 讓同一個模型同時測試多種 local scales。

In [ ]:
class WordMCNN(nn.Module):
    def __init__(self,vocab,emb_dim=128,windows=(2,3,4,5),filters=64,dropout=.3):
        super().__init__();self.embedding=nn.Embedding(vocab,emb_dim,padding_idx=PAD)
        self.windows=windows;self.convs=nn.ModuleList([nn.Conv1d(emb_dim,filters,k) for k in windows])
        self.drop=nn.Dropout(dropout);self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,ids,return_maps=False):
        x=self.embedding(ids).transpose(1,2);maps=[F.relu(c(x)) for c in self.convs]
        feats=[F.adaptive_max_pool1d(m,1).squeeze(-1) for m in maps]
        out=self.fc(self.drop(torch.cat(feats,1))).squeeze(1)
        return (out,maps) if return_maps else out
mcnn=fit_word(WordMCNN(len(itos)),epochs=2 if FAST_MODE else 6)

## 11. 用 top-activating n-grams 解讀一個 learned filter

這是保守的 interpretability exercise：找出某個 filter activation 最大的 local windows。

In [ ]:
def top_windows(model,text,window_index=1,filter_index=0,topn=8):
    toks=simple_tokenize(text)[:MAX_LEN]; ids=encode(text).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad(): _,maps=model(ids,return_maps=True)
    fmap=maps[window_index][0,filter_index].cpu().numpy(); k=model.windows[window_index]
    order=np.argsort(-fmap)[:topn]
    return [(float(fmap[i]),' '.join(toks[i:i+k])) for i in order if i+k<=len(toks)]
for score,ng in top_windows(mcnn,val_ds[0]['text']): print(round(score,3),ng)

# Part II — DistilBERT contextual representations

In [ ]:
MODEL_NAME='distilbert/distilbert-base-uncased'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
encoder=AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
for p in encoder.parameters(): p.requires_grad=False
BERT_MAX_LEN=192
sample=tokenizer('The movie was not very good, although the acting was excellent.',return_tensors='pt')
print(tokenizer.convert_ids_to_tokens(sample['input_ids'][0]))
with torch.no_grad(): h=encoder(**{k:v.to(DEVICE) for k,v in sample.items()}).last_hidden_state
print('hidden state shape',h.shape)

## 12. Frozen DistilBERT + MCNN

先 freeze encoder，形成乾淨的 representation-learning 比較：只有 downstream local-pattern detector 從 IMDb labels 學習。

In [ ]:
class BertDS(Dataset):
    def __init__(self,ds):self.ds=ds
    def __len__(self):return len(self.ds)
    def __getitem__(self,i):
        r=self.ds[i];e=tokenizer(r['text'],truncation=True,max_length=BERT_MAX_LEN,padding='max_length',return_tensors='pt')
        return e['input_ids'][0],e['attention_mask'][0],torch.tensor(float(r['label']))
def bert_loader(ds,bs=16,shuffle=False):return DataLoader(BertDS(ds),batch_size=bs,shuffle=shuffle)
btr=bert_loader(train_ds,16,True);bva=bert_loader(val_ds,32);bte=bert_loader(test_ds,32)
class BertMCNN(nn.Module):
    def __init__(self,encoder,hidden=768,windows=(2,3,4,5),filters=64,dropout=.3):
        super().__init__();self.encoder=encoder;self.convs=nn.ModuleList([nn.Conv1d(hidden,filters,k) for k in windows]);self.drop=nn.Dropout(dropout);self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,ids,mask):
        with torch.no_grad(): h=self.encoder(input_ids=ids,attention_mask=mask).last_hidden_state
        h=h*mask.unsqueeze(-1);x=h.transpose(1,2);feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs]
        return self.fc(self.drop(torch.cat(feats,1))).squeeze(1)
bert_mcnn=BertMCNN(encoder).to(DEVICE)

In [ ]:
def eval_bert(model,loader):
    model.eval();ys=[];ps=[]
    with torch.no_grad():
        for ids,mask,y in loader:
            p=torch.sigmoid(model(ids.to(DEVICE),mask.to(DEVICE))).cpu().numpy();ys.extend(y.numpy());ps.extend(p)
    return np.array(ys),np.array(ps)
def fit_bert(model,epochs=2,lr=1e-3):
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr);lossfn=nn.BCEWithLogitsLoss();best=None;best_auc=-1
    for ep in range(epochs):
        model.train()
        for ids,mask,y in btr:
            ids,mask,y=ids.to(DEVICE),mask.to(DEVICE),y.to(DEVICE);opt.zero_grad(set_to_none=True);loss=lossfn(model(ids,mask),y);loss.backward();opt.step()
        yy,pp=eval_bert(model,bva);m=metric_dict(yy,pp);print(ep+1,m)
        if m['roc_auc']>best_auc:best_auc=m['roc_auc'];best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best);return model
bert_mcnn=fit_bert(bert_mcnn,epochs=1 if FAST_MODE else 3)

## 13. Optional：完整 DistilBERT fine-tuning

使用更小 learning rate（例如 2e-5）。刻意把這個 extension 與 frozen feature experiment 分開，避免把 representation quality 與 end-to-end adaptation 混在一起。

In [ ]:
# Optional starter (not run automatically):
# clf=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2).to(DEVICE)
# optimizer=torch.optim.AdamW(clf.parameters(),lr=2e-5)
# Then train with CrossEntropyLoss on input_ids/attention_mask.

## 14. 最後 Test Comparison

到這裡才對選好的模型使用 held-out test subset。

In [ ]:
yw,pw=eval_word(mcnn,te_loader)
yb,pb=eval_bert(bert_mcnn,bte)
# TF-IDF test
Xte=tfidf.transform(test_ds['text']);pt=tfidf_model_p=logreg.predict_proba(Xte)[:,1]
display(pd.DataFrame([
 {'model':'TF-IDF + logistic regression',**metric_dict(np.array(test_ds['label']),pt)},
 {'model':'learned embedding + MCNN',**metric_dict(yw,pw)},
 {'model':'frozen DistilBERT + MCNN',**metric_dict(yb,pb)}]))

## 15. Error Analysis 與 Challenge Sentences

In [ ]:
errs=[]
for txt,y,p in zip(test_ds['text'],yb,pb):
    pred=int(p>=.5)
    if pred!=int(y):errs.append({'true':int(y),'p_positive':float(p),'confidence':float(max(p,1-p)),'text':txt[:600]})
display(pd.DataFrame(errs).sort_values('confidence',ascending=False).head(15))
challenge=[
 'I wanted to love this film, but it was not good at all.',
 'Great acting. Shame about everything else.',
 'This was so bad that it somehow became entertaining.',
 'Not perfect, but much better than I expected.'
]
enc=tokenizer(challenge,truncation=True,max_length=BERT_MAX_LEN,padding=True,return_tensors='pt').to(DEVICE)
with torch.no_grad(): probs=torch.sigmoid(bert_mcnn(enc['input_ids'],enc['attention_mask'])).cpu().numpy()
for s,p in zip(challenge,probs):print(round(float(p),3),s)

# 銜接 Protein Language Model

你現在已經具備後面研究真正需要的 computation pattern：

```text
text → DistilBERT → B×L×768 → MCNN → classification
protein → ESM-2/ProtT5 → B×L×D → MCNN → biological prediction
```

下一個 notebook 主要是換 alphabet 與 encoder，downstream 邏輯基本不變。